# news_sector_hot_issue — 시간대를 정하면 업종별 핫이슈 1줄 정리

`news_macro_hot_issue_tmp.ipynb`와 같은 과정을 **업종(`sectors`)마다** 한다. 업종마다 정한 시간대의 사건 기사 중 **매체 수가 가장 높은 사건 1개**를 골라 OpenAI로 한 줄씩 정리한다.

```
news_extract  ─ llm_source.jsonl에 기사 한 줄(sectors: 한국 WI26 업종 0~3개)
news_track    ─ 같은 줄에 story_id(사건 번호), news_outlets(소식 매체 수) / importance.csv에 story_outlets(사건 매체 수)
   ▼
news_sector_hot_issue_tmp.ipynb ─ 시간대 거르기 → 사건 기사만 → 업종마다: 사건마다 하나 → 매체 수 1위 → OpenAI 1줄(한 번 호출로 모든 업종)
   ▼  sector_hot_issues.jsonl (실행마다 한 줄: 공통 칸 + 업종 목록)
반도체: ~
은행: ~
```

- 이 노트북은 `llm_source.jsonl`, `importance.csv`를 **읽기만** 한다.
- 프롬프트: `prompt/sector_hot_issue_prompt.md`(proj/prompt에서 복제).

## 1. 설정

- `START_KST`, `END_KST`: 정리할 시간대(한국 시각, `START_KST` 이상 `END_KST` 미만).
- `TIME_COLUMN`: 시간대를 어느 시각으로 거를지. `published_utc`(기사 발행 시각) 또는 `first_seen_utc`(우리가 처음 수집한 시각).
- `SECTORS`: 정리할 업종 목록. `None`이면 시간대 안 사건 기사에 나온 업종 모두(`SECTOR_ORDER` 순서).
- `MAX_ARTICLES_PER_ISSUE`: 업종마다 LLM에 넘길 기사 수(대표 기사 포함).
- `RUN_LLM = False`면 업종별로 고른 사건과 LLM에 넘길 입력만 보여 주고 **OpenAI를 부르지 않는다.**
- 키는 proj 폴더 `.env`의 `OPENAI_API_KEY`이며 값은 출력하지 않는다.
- 순위에 쓰는 매체 수는 **정한 시간대 안의 기사만으로** 다시 센다(2026-11-20 바꿈). `news_track`이 기사마다 남긴 센 출처로 센다. 전체 기간 매체 수는 결과에 `*_all`로 함께 남긴다.
- 처음에 수집 DB(`.env`의 `GNEWS_DB_URL`)에서 시간대 안 기사 수를 세어, `news_extract`가 아직 처리하지 않은 기사가 몇 건인지 알린다(요약은 멈추지 않는다).
- `.env`는 노트북 폴더에서 위로 올라가며 처음 나오는 것을 쓴다.

In [ ]:
import datetime as dt
import json
import urllib.error
import urllib.request
from pathlib import Path

import sqlalchemy as sa

import pandas as pd
from IPython.display import display

# ===== 여기만 바꿔서 쓴다 =====
ROOT = Path.cwd()                      # news_search 폴더(이 노트북이 있는 폴더에서 연다)
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])   # .env가 있는 폴더(노트북 폴더에서 위로 올라가며 처음 나오는 곳)
RESULT_DIR = ROOT / 'source' / 'live'  # news_extract·news_track이 함께 쓰는 결과 폴더
SECTOR_PROMPT_PATH = ROOT / 'prompt' / 'sector_hot_issue_prompt.md'   # proj/prompt에서 복제
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 OpenAI로 업종별 1줄 정리를 한다
START_KST = '2026-11-18 16:00'   # 시간대 시작(한국 시각, 포함)
END_KST = '2026-11-19 07:30'     # 시간대 끝(한국 시각, 미포함)
TIME_COLUMN = 'published_utc'    # 'published_utc'(발행 시각) 또는 'first_seen_utc'(처음 수집한 시각)
SECTORS = None                   # 예: ['반도체', '은행']. None이면 시간대 안에 나온 업종 모두
MAX_ARTICLES_PER_ISSUE = 3
# ==============================

# 한국 WI26 업종(리스트 추출 프롬프트 graph.sectors 목록과 같은 순서)
SECTOR_ORDER = ['IT가전', 'IT하드웨어', '건강관리', '건설', '기계', '디스플레이', '미디어,교육', '반도체', '보험', '비철금속',
                '상사,자본재', '소매(유통)', '소프트웨어', '에너지', '운송', '유틸리티', '은행', '자동차', '조선', '증권', '철강',
                '통신서비스', '필수소비재', '호텔,레저', '화장품,의류', '화학']
DB_ENV = 'GNEWS_DB_URL'   # 수집 DB 주소(.env 이름). 시간대 안 기사 중 아직 처리 안 된 수를 알리는 데만 쓴다
ARTICLES_PATH = RESULT_DIR / 'articles.jsonl'       # news_extract가 처리한 기사(DB 확인에 key만 읽는다)
LLM_SOURCE_PATH = RESULT_DIR / 'llm_source.jsonl'   # 읽기만 한다
IMPORTANCE_PATH = RESULT_DIR / 'importance.csv'     # news_track의 매체 수(사건 매체 수 story_outlets를 읽기만 한다)
SECTOR_ISSUES_PATH = RESULT_DIR / 'sector_hot_issues.jsonl'   # 정리할 때마다 업종마다 한 줄
USAGE_PATH = RESULT_DIR / 'llm_usage.csv'           # LLM 호출마다 토큰 수(news_extract·news_track과 함께 씀)
KST = dt.timezone(dt.timedelta(hours=9))
assert ROOT.name == 'news_search', 'news_search 폴더에서 열어야 한다.'

## 2. 작은 도구

`news_macro_hot_issue_tmp.ipynb`와 같은 도구다.
- `env_value`: proj 폴더 `.env`에서 키를 읽는다(값은 출력하지 않는다).
- `ask_openai`: OpenAI Responses API. 오류가 나면 오류 종류·코드만 알린다.

In [ ]:
def env_value(name):
    env_file = PROJ / '.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


def read_jsonl(path):
    # 한 줄씩 읽는다. 다른 노트북이 쓰는 중인 마지막 줄(덜 쓰인 줄)은 건너뛴다.
    rows = []
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def append_jsonl(path, rows):
    with path.open('a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def append_csv(rows, path):
    if rows:
        data = pd.DataFrame(rows)
        if path.exists():   # 이미 있는 파일의 열 순서에 맞춘다
            data = data.reindex(columns=pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
        data.to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')


class LLMError(Exception):
    pass


def ask_openai(instructions, user_message, text_format):
    request = urllib.request.Request(
        'https://api.openai.com/v1/responses', method='POST',
        headers={'Authorization': f"Bearer {env_value('OPENAI_API_KEY')}", 'Content-Type': 'application/json'},
        data=json.dumps({'model': LLM_MODEL, 'instructions': instructions, 'input': user_message, 'store': False,
                         'reasoning': {'effort': 'medium'}, 'text': {'format': text_format}}).encode('utf-8'))
    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode('utf-8', 'replace')).get('error', {})
        except ValueError:
            err = {}
        raise LLMError(f"OpenAI HTTP {e.code} type={err.get('type')} code={err.get('code')}") from None
    text = ''.join(c.get('text', '') for item in body.get('output', []) if item.get('type') == 'message'
                   for c in item.get('content', []))
    return text, body.get('usage') or {}


def log_usage(step, ref, usage):
    append_csv([{'time_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'), 'step': step, 'ref': ref,
                 'input_tokens': usage.get('input_tokens'),
                 'cached_tokens': (usage.get('input_tokens_details') or {}).get('cached_tokens'),
                 'output_tokens': usage.get('output_tokens')}], USAGE_PATH)


print('OPENAI_API_KEY:', '설정됨' if env_value('OPENAI_API_KEY') else '미설정')

## 3. 업종마다 사건 고르기

1. `llm_source.jsonl`에서 `TIME_COLUMN`이 시간대 안이고 `story_id`가 있는 기사(사건 기사)만 남긴다. 분석·추천 글처럼 사건으로 묶이지 않은 기사는 `news_outlets`가 없어 뺀다.
2. 순위용 매체 수(`news_outlets` 소식 매체 수, `story_outlets` 사건 매체 수)는 **시간대 안 기사로만 다시 센다**(2026-11-20 바꿈, 전체 기간 값은 `*_all`).
3. 업종마다(`sectors`에 그 업종이 있는 기사만):
   - 사건마다 한 줄: `news_outlets`가 가장 큰 기사(같으면 최근 발행)를 대표로 한다.
   - 사건 순위: `news_outlets` 큰 순 → `story_outlets` 큰 순 → 대표 기사 발행 시각 최근 순. **1위 사건 하나**를 고른다.
4. 같은 사건이 여러 업종의 1위가 될 수 있다(예: 한 기사가 반도체·소프트웨어 둘 다).
5. LLM 입력: 업종마다 대표 기사부터 `MAX_ARTICLES_PER_ISSUE`건(그 업종의 시간대 안 기사)의 제목·매체·발행 시각·short_summary·comparison_summary.

In [ ]:
def to_utc(text):
    t = pd.Timestamp(text)
    return (t.tz_localize(KST) if t.tzinfo is None else t).tz_convert('UTC')


def db_coverage(start_kst, end_kst):
    """DB(수집 결과)에서 시간대 안 기사(거르기 1단계 통과) 중 news_extract가 아직 처리하지 않은 수를 알린다(2026-11-20 더함).
    요약은 멈추지 않는다. 빠진 기사가 있으면 news_extract → news_track을 먼저 돌리면 빠짐없이 요약된다."""
    url = env_value(DB_ENV)
    if not url:
        print(f'DB 확인 건너뜀: .env에 {DB_ENV}가 없다.')
        return None
    col = {'published_utc': 'published_utc', 'first_seen_utc': 'first_seen_utc'}[TIME_COLUMN]
    engine = sa.create_engine(url)
    try:
        with engine.connect() as con:
            keys = {k for (k,) in con.execute(sa.text(f'select key from items where coalesce("drop", \'\') = \'\' and {col} >= :s and {col} < :e and first_seen_utc < :e'),
                                              {'s': to_utc(start_kst).to_pydatetime(), 'e': to_utc(end_kst).to_pydatetime()})}
    except Exception as e:
        print('DB 확인 실패(요약은 계속):', type(e).__name__, str(e)[:150])
        return None
    finally:
        engine.dispose()
    done = {r.get('key') for r in read_jsonl(ARTICLES_PATH)}
    todo = len(keys - done)
    print(f'DB 확인: 시간대 안 수집 기사 {len(keys)}건 중 news_extract 처리 {len(keys & done)}건, 아직 처리 안 됨 {todo}건'
          + (' — news_extract → news_track을 먼저 돌리면 빠짐없이 요약된다' if todo else ''))
    return todo


def window_outlets(rows, in_window):
    """순위용 매체 수를 시간대 안 기사로만 다시 센다(2026-11-20 바꿈).
    news_track이 기사마다 남긴 센 출처(source_key)와 소식 번호(news_id)로, 시간대 안 기사만 세어
    news_outlets(소식 매체 수)·story_outlets(사건 매체 수)를 바꾼다. 전체 기간 값은 news_outlets_all·story_outlets_all로 남긴다."""
    rows = rows.copy()
    rows['news_outlets_all'] = pd.to_numeric(rows['news_outlets'], errors='coerce')
    if IMPORTANCE_PATH.exists():
        imp = pd.read_csv(IMPORTANCE_PATH, dtype=str).drop_duplicates('story_id').set_index('story_id')
        rows['story_outlets_all'] = pd.to_numeric(rows['story_id'].map(imp['story_outlets']), errors='coerce')
    else:
        rows['story_outlets_all'] = float('nan')
    if 'source_key' not in rows or rows['source_key'].isna().all():
        print('주의: llm_source.jsonl에 센 출처(source_key)가 없다 — news_track을 한 번 돌려야 한다. 지금은 전체 기간 매체 수로 순위를 매긴다.')
        rows['story_outlets'] = rows['story_outlets_all']
        return rows
    counted = rows[in_window & rows['source_key'].fillna('').astype(str).ne('')]
    news = counted.groupby(['story_id', 'news_id'])['source_key'].nunique()
    story = counted.groupby('story_id')['source_key'].nunique()
    rows['news_outlets'] = [news.get((s, n)) for s, n in zip(rows['story_id'], rows['news_id'])]
    rows['story_outlets'] = rows['story_id'].map(story)
    return rows


def to_int(values):
    return pd.to_numeric(pd.Series(values), errors='coerce').fillna(0).astype(int)


def int_or0(value):
    try:
        return 0 if value is None or pd.isna(value) else int(value)
    except (TypeError, ValueError):
        return 0


def to_kst_text(utc_text):
    """UTC ISO 시각 → 한국 시각 ISO 문자열(예: 2026-11-16T05:42:00+09:00). 못 읽으면 빈 값."""
    t = pd.to_datetime(utc_text, utc=True, errors='coerce')
    return '' if pd.isna(t) else t.tz_convert(KST).isoformat(timespec='seconds')


def text_or(value, default=''):
    return value if isinstance(value, str) and value else default


def pick_sector_issues(start_kst=None, end_kst=None, sectors=None):
    """시간대 안 사건 기사(events)와 업종마다 1위 사건의 대표 기사(top, 업종 순서)를 돌려준다."""
    start_kst, end_kst = start_kst or START_KST, end_kst or END_KST
    sectors = sectors if sectors is not None else SECTORS
    db_coverage(start_kst, end_kst)
    rows = pd.DataFrame(read_jsonl(LLM_SOURCE_PATH))
    if rows.empty:
        return rows, rows
    for c in ['sectors', 'story_id', 'news_outlets']:
        if c not in rows:
            rows[c] = None
    if rows['story_id'].isna().all():
        print('주의: story_id가 비어 있다 — news_track을 먼저 돌려야 한다.')
    rows['time'] = pd.to_datetime(rows[TIME_COLUMN], utc=True, format='ISO8601', errors='coerce')
    in_window = rows['time'].ge(to_utc(start_kst)) & rows['time'].lt(to_utc(end_kst))
    # 끝 시각까지 수집된 기사만(2026-11-23): 발행 시각 기준으로 나중에 다시 돌려도, 늦게 수집된 기사가 섞이지 않고 정규 실행(06:20)과 같은 기사를 쓴다
    in_window &= pd.to_datetime(rows['first_seen_utc'], utc=True, format='ISO8601', errors='coerce').lt(to_utc(end_kst))
    rows = window_outlets(rows, in_window)      # 순위용 매체 수: 시간대 안 기사로만
    has_story = rows['story_id'].fillna('').astype(str) != ''
    events = rows[in_window & has_story].copy()
    events['sectors'] = events['sectors'].map(lambda x: x if isinstance(x, list) else [])
    events['news_outlets'] = pd.to_numeric(events['news_outlets'], errors='coerce').fillna(0).astype(int)
    events['story_outlets'] = to_int(events['story_outlets'])
    events = events.sort_values(['news_outlets', 'story_outlets', 'time'], ascending=[False, False, False]).reset_index(drop=True)

    found = {s for x in events['sectors'] for s in x}
    order = [s for s in SECTOR_ORDER if s in found] + sorted(found - set(SECTOR_ORDER))
    wanted = order if sectors is None else [s for s in sectors if s in found]
    missing = [] if sectors is None else [s for s in sectors if s not in found]
    tops = []
    for s in wanted:
        cand = events[events['sectors'].map(lambda x: s in x)]
        r = cand.iloc[0].copy()             # 이미 순위 순으로 정렬돼 있어 맨 앞이 1위 사건의 대표 기사
        r['sector'], r['sector_stories'] = s, cand['story_id'].nunique()
        tops.append(r)
    top = pd.DataFrame(tops).reset_index(drop=True) if tops else pd.DataFrame()
    print(f'시간대 {start_kst} ~ {end_kst} (KST, {TIME_COLUMN}) | 시간대 안 기사 {int(in_window.sum())}건, '
          f'그중 사건 기사 {len(events)}건 / 업종 {len(order)}개 → 정리할 업종 {len(top)}개'
          + (f' (기사가 없는 업종: {missing})' if missing else ''))
    return events, top


def sector_input(events, top):
    """LLM에 넘길 업종 목록(업종 순서)."""
    items = []
    for _, r in top.iterrows():
        same = events[(events['story_id'] == r['story_id']) & events['sectors'].map(lambda x: r['sector'] in x)]
        same = same.head(MAX_ARTICLES_PER_ISSUE)   # 대표 기사가 맨 앞
        items.append({'sector': r['sector'], 'news_outlets': int(r['news_outlets']), 'story_outlets': int(r['story_outlets']),
                      'articles': [{'title': a['title'], 'outlet': text_or(a.get('provider'), a['outlet']),
                                    'published_utc': a['published_utc'], 'short_summary': text_or(a.get('short_summary')),
                                    'comparison_summary': text_or(a.get('comparison_summary'))}
                                   for _, a in same.iterrows()]})
    return items

## 4. 업종별 1줄 정리

- 업종마다 고른 사건을 **한 번의 호출**로 OpenAI에 넘겨 `{"lines": [{"sector", "line"}, ...]}`(업종마다 한 줄)을 받는다.
- 받은 줄을 `업종: 문장`으로 보여 주고 `sector_hot_issues.jsonl`에 **실행마다 한 줄** 남긴다.
  - 공통 칸: 브리핑 날짜 `date`(시간대 끝의 한국 날짜), 생성 시각 `created_kst`(한국), 시간대 `start_kst`·`end_kst`, `time_column`, `model`
  - `sector_issues`: 업종마다 `{sector, story_id(고른 사건), key(대표 기사), published_kst(대표 기사 발행 시각, 한국), news_outlets, story_outlets, sector_stories(업종 안 사건 수), line(1줄)}`
- 받은 업종 이름이 입력과 다르거나 빠지면 알리고, 그 업종은 저장하지 않는다. 고른 업종이 없으면 OpenAI를 부르지 않는다.

In [ ]:
SECTOR_PROMPT = SECTOR_PROMPT_PATH.read_text(encoding='utf-8')


def sector_hot_issues(start_kst=None, end_kst=None, sectors=None):
    start_kst, end_kst = start_kst or START_KST, end_kst or END_KST
    events, top = pick_sector_issues(start_kst, end_kst, sectors)
    if top.empty:
        print('정리할 업종 이슈가 없다.')
        return {}
    display(top[['sector', 'story_id', 'news_outlets', 'story_outlets', 'sector_stories', 'published_utc', 'title']])
    items = sector_input(events, top)
    if not RUN_LLM:
        print('RUN_LLM = False라 OpenAI를 부르지 않는다. LLM에 넘길 입력:')
        print(json.dumps({'sectors': items}, ensure_ascii=False, indent=1))
        return {}
    text, usage = ask_openai(SECTOR_PROMPT, 'sectors json:\n' + json.dumps({'sectors': items}, ensure_ascii=False), {'type': 'json_object'})
    log_usage('sector_hot_issue', f'{start_kst}~{end_kst}', usage)
    got = {}
    for x in json.loads(text).get('lines', []):
        if isinstance(x, dict) and x.get('sector'):
            got[str(x['sector']).strip()] = str(x.get('line', '')).strip().lstrip('-•· ').strip()
    wrong = sorted(set(got) - set(top['sector'])) + [s for s in top['sector'] if not got.get(s)]
    if wrong:
        print('주의: 입력과 맞지 않거나 빠진 업종(저장하지 않음):', wrong)
    now = dt.datetime.now(dt.timezone.utc)
    base = {'date': to_utc(end_kst).tz_convert(KST).strftime('%Y-%m-%d'),   # 브리핑 날짜(시간대 끝의 한국 날짜)
            'created_kst': now.astimezone(KST).strftime('%Y-%m-%d %H:%M:%S'),   # 생성 시각(한국)
                'start_kst': start_kst, 'end_kst': end_kst, 'time_column': TIME_COLUMN, 'model': LLM_MODEL}
    items = [{'sector': r['sector'], 'story_id': r['story_id'], 'key': r['key'], 'published_kst': to_kst_text(r['published_utc']),
              'news_outlets': int(r['news_outlets']),
              'story_outlets': int(r['story_outlets']), 'sector_stories': int(r['sector_stories']),
              'news_outlets_all': int_or0(r['news_outlets_all']), 'story_outlets_all': int_or0(r['story_outlets_all']),
              'line': got[r['sector']]}
             for _, r in top.iterrows() if got.get(r['sector'])]
    append_jsonl(SECTOR_ISSUES_PATH, [{**base, 'sector_issues': items}])   # 실행마다 한 줄: 공통 칸 + 업종 목록
    for r in items:
        print(f"{r['sector']}: {r['line']}")
    return {r['sector']: r['line'] for r in items}

## 5. 실행

In [ ]:
sector_lines = sector_hot_issues()